# AI Resume Screening & Candidate Ranking System

This notebook demonstrates the end-to-end Machine Learning workflow for building a Resume Screening System. We will cover:
1. Data Loading & Exploration
2. Text Cleaning & NLP Preprocessing
3. Skill Extraction using predefined dictionaries and spaCy
4. Feature Extraction using TF-IDF Vectorization
5. Cosine Similarity Calculation
6. Candidate Ranking & Visualization

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import spacy

# Ensure NLTK data is downloaded
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

sns.set_theme(style="whitegrid")

## 2. Load Sample Data
For this demonstration, we'll use a small subset of the Kaggle Resume dataset and a sample Job Description.

In [ ]:
# Try to load from the CSV file
try:
    df_resumes = pd.read_csv('../Resume/Resume.csv').head(10)
    print(f"Loaded {len(df_resumes)} resumes.")
    text_col = 'Resume_str' if 'Resume_str' in df_resumes.columns else df_resumes.columns[0]
    resumes = df_resumes[text_col].tolist()
except Exception as e:
    print("Could not load CSV, using mock data.")
    resumes = [
        "Experienced Software Engineer with 5 years of Python, SQL, and Machine Learning experience. Skilled in React and AWS.",
        "Data Analyst with strong SQL, Excel, and Tableau skills. Familiar with Python for data manipulation.",
        "Frontend Developer proficient in HTML, CSS, JavaScript, and React. Good eye for design."
    ]

job_description = "We are looking for a Data Scientist or Machine Learning Engineer with strong Python and SQL skills. Experience with Machine Learning models and AWS is a huge plus."

print("\nJob Description:")
print(job_description)

## 3. Text Preprocessing

In [ ]:
def clean_and_preprocess(text):
    if not isinstance(text, str):
        text = str(text)
    
    # Clean
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove Stopwords and Lemmatize
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    processed = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 2]
    
    return ' '.join(processed)

jd_processed = clean_and_preprocess(job_description)
resumes_processed = [clean_and_preprocess(r) for r in resumes]

print("Processed JD:", jd_processed)
print("Processed Resume 1:", resumes_processed[0][:100], "...")

## 4. Skill Extraction

In [ ]:
SKILLS_LIST = ["python", "sql", "machine learning", "aws", "react", "excel", "tableau", "html", "css", "javascript", "data analysis"]

def extract_skills(text):
    text_lower = text.lower()
    found_skills = []
    for skill in SKILLS_LIST:
        if re.search(r'\b' + re.escape(skill) + r'\b', text_lower):
            found_skills.append(skill)
    return found_skills

jd_skills = extract_skills(job_description)
print("JD Skills Required:", jd_skills)

resume_skills_list = [extract_skills(r) for r in resumes]

## 5. Scoring: TF-IDF & Cosine Similarity

In [ ]:
results = []

for i, resume_proc in enumerate(resumes_processed):
    # Calculate Similarity
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([jd_processed, resume_proc])
    cosine_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
    sim_score = cosine_sim * 100
    
    # Calculate Skill Match
    candidate_skills = resume_skills_list[i]
    matched_skills = set(jd_skills).intersection(set(candidate_skills))
    skill_score = (len(matched_skills) / len(jd_skills) * 100) if len(jd_skills) > 0 else 0
    
    # Final Score
    final_score = (0.7 * sim_score) + (0.3 * skill_score)
    
    results.append({
        'Candidate': f'Candidate {i+1}',
        'Similarity (%)': round(sim_score, 2),
        'Skill Match (%)': round(skill_score, 2),
        'Final Score (%)': round(final_score, 2),
        'Matched Skills': ", ".join(matched_skills)
    })

df_results = pd.DataFrame(results).sort_values(by='Final Score (%)', ascending=False).reset_index(drop=True)
df_results

## 6. Visualization

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Final Score (%)', y='Candidate', data=df_results, palette='viridis')
plt.title('Candidate Final Scores')
plt.xlim(0, 100)
plt.show()